In [1]:
import pandas as pd
import sys
sys.path.append('..')

from src.cointegration import find_cointegrated_pairs, test_cointegration

In [4]:
# Load data
prices = pd.read_csv("data/raw/prices.csv", parse_dates=['Date'])

# Find cointegrated pairs
pairs = find_cointegrated_pairs(prices, max_pairs=20)

print("\n=== TOP COINTEGRATED PAIRS ===")
print(pairs.head(10))


Testing 300 pairs for cointegration...


100%|███████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:24<00:00, 12.15it/s]


Found 9 cointegrated pairs (p < 0.05)

=== TOP COINTEGRATED PAIRS ===
  Ticker1 Ticker2   P_Value  Hedge_Ratio
8     LOW      MS  0.005500     2.166614
2     CVX     EOG  0.010744     1.014278
0    AAPL     LOW  0.018035     0.991151
6     JNJ      MS  0.020162     0.653622
3   GOOGL      HD  0.021551     0.530862
4      HD      MS  0.032896     2.573531
1     COP     EOG  0.039485     0.979965
7     JNJ     UNH  0.040455     0.131513
5     JNJ     LOW  0.044564     0.285593


In [6]:
# Save for later use
pairs.to_csv("data/processed/cointegrated_pairs.csv", index=False)
print("\n✓ Saved pairs to data/processed/cointegrated_pairs.csv")


✓ Saved pairs to data/processed/cointegrated_pairs.csv


In [ ]:
# collect_and_test_fundamentals.py
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

# 1. Collect data
def collect_fundamentals(tickers):
    data = []
    for ticker in tickers:
        stock = yf.Ticker(ticker)
        info = stock.info
        data.append({
            'Ticker': ticker,
            'Sector': info.get('sector'),
            'PE_Ratio': info.get('trailingPE'),
            'ROE': info.get('returnOnEquity'),
            'Market_Cap': info.get('marketCap'),
        })
    return pd.DataFrame(data)

# 2. Calculate similarity
def sector_match(t1, t2, df):
    s1 = df[df['Ticker']==t1]['Sector'].values[0]
    s2 = df[df['Ticker']==t2]['Sector'].values[0]
    return 1.0 if s1 == s2 else 0.0

# 3. Test
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JNJ', 'PFE']  # Sample
fundamentals = collect_fundamentals(tickers)
fundamentals.to_csv('fundamentals_sample.csv')

print(f"\nSector similarity AAPL-MSFT: {sector_match('AAPL', 'MSFT', fundamentals)}")
print(f"Sector similarity AAPL-JNJ: {sector_match('AAPL', 'JNJ', fundamentals)}")